In [1]:
import pickle
import pandas as pd
import os

In [2]:
import pandas as pd
import ast

def construct_identifier(article_pii, table_no):
    if pd.isna(article_pii) or pd.isna(table_no):
        return None
    article_pii_str = str(article_pii)
    table_no_str = str(table_no)
    if '[' in table_no_str and ',' in table_no_str:
        try:
            table_no_clean = table_no_str.strip('[]').replace(' ', '').replace(',', '_')
            return f"{article_pii_str}_{table_no_clean}"
        except:
            return f"{article_pii_str}_{table_no_str}"
    else:
        table_no_clean = table_no_str.strip('[]')
        return f"{article_pii_str}_{table_no_clean}"

def get_pii_only(identifier):
    if identifier is None:
        return None
    parts = str(identifier).split('_')
    return parts[0]

def parse_composition(comp_str):
    try:
        if pd.isna(comp_str) or comp_str == '[]':
            return []
        compositions = ast.literal_eval(comp_str)
        normalized_compositions = []
        for comp in compositions:
            if len(comp) >= 3:
                element, value, unit = comp[0], comp[1], comp[2]
                if unit == 'at':
                    unit = 'mol'
                normalized_compositions.append((element, value, unit))
            else:
                normalized_compositions.append(comp)
        return normalized_compositions
    except:
        return []

def parse_property(prop_str):
    try:
        if pd.isna(prop_str):
            return ()
        return ast.literal_eval(prop_str)
    except:
        return ()

def normalize_compounds(tuples_list):
    from collections import defaultdict
    grouped_data = defaultdict(list)
    for tup in tuples_list:
        grouped_data[tup[0]].append(tup)
    
    normalized_tuples = []
    for material_id, compounds in grouped_data.items():
        total_value = sum(comp[2] for comp in compounds)
        multiplier = 100 if total_value <= 1.05 else 1
        
        for comp in compounds:
            normalized_tuple = (
                comp[0], comp[1], round(comp[2] * multiplier, 1), comp[3]
            )
            normalized_tuples.append(normalized_tuple)
    
    return normalized_tuples

def prepare_data(df):
    df = df.copy()
    df['Identifier'] = df.apply(lambda row: construct_identifier(row['Article PII'], row['Table No.']), axis=1)
    df['Parsed_Composition'] = df['Composition'].apply(parse_composition)
    df['Parsed_Property'] = df['Property'].apply(parse_property)
    return df

def create_comp_list(df):
    comp_data = []
    for _, row in df.iterrows():
        if row['Parsed_Composition'] and pd.notna(row['Identifier']):
            comp_tuples = [(row['Identifier'], comp[0], comp[1], comp[2]) for comp in row['Parsed_Composition']]
            normalized = normalize_compounds(comp_tuples)
            
            compositions = []
            for comp in normalized:
                if comp[0] == row['Identifier']:
                    compositions.append((comp[1], comp[2], comp[3]))
            
            if compositions:
                sorted_compositions = tuple(sorted(compositions))
                pii_only = get_pii_only(row['Identifier'])
                comp_data.append([pii_only, sorted_compositions])
    
    return comp_data

def create_prop_list(df):
    prop_data = []
    for _, row in df.iterrows():
        if row['Parsed_Property'] and len(row['Parsed_Property']) == 3 and pd.notna(row['Identifier']):
            pii_only = get_pii_only(row['Identifier'])
            prop_data.append([pii_only, row['Parsed_Property']])
    return prop_data

def create_comp_and_prop_list(df):
    comp_and_prop_data = []
    for _, row in df.iterrows():
        if (row['Parsed_Composition'] and row['Parsed_Property'] and 
            len(row['Parsed_Property']) == 3 and pd.notna(row['Identifier'])):
            
            comp_tuples = [(row['Identifier'], comp[0], comp[1], comp[2]) for comp in row['Parsed_Composition']]
            normalized = normalize_compounds(comp_tuples)
            
            compositions = []
            for comp in normalized:
                if comp[0] == row['Identifier']:
                    compositions.append((comp[1], comp[2], comp[3]))
            
            if compositions:
                sorted_compositions = tuple(sorted(compositions))
                combined_data = (sorted_compositions, row['Parsed_Property'])
                pii_only = get_pii_only(row['Identifier'])
                comp_and_prop_data.append([pii_only, combined_data])
    
    return comp_and_prop_data

def deduplicate_data(data_list, data_type="generic"):
    unique_data = []
    seen = set()
    
    for item in data_list:
        if item:
            hashable_item = tuple(tuple(subitem) if isinstance(subitem, (list, tuple)) else subitem for subitem in item)
            if hashable_item not in seen:
                seen.add(hashable_item)
                unique_data.append(item)
    
    return unique_data

def match_tuple_comp(p, g):
    if not p or not g or len(p) != 2 or len(g) != 2:
        return False
    return p[0] == g[0] and p[1] == g[1]

def match_tuple_prop(p, g):
    if not p or not g or len(p) != 2 or len(g) != 2:
        return False
    if p[0] is None or g[0] is None:
        return False
    
    try:
        p_val = p[1][1]
        g_val = g[1][1]
        magnitude = max(abs(p_val), abs(g_val))
        if magnitude < 1e-6:
            tolerance = magnitude * 0.01
        else:
            tolerance = 0.01
        matches = abs(p_val - g_val) < tolerance
        return (p[0] == g[0] and 
                p[1][0] == g[1][0] and
                matches and
                p[1][2] == g[1][2])
    except (TypeError, IndexError):
        return False

def match_tuple_comp_and_prop(p, g):
    if not p or not g or len(p) != 2 or len(g) != 2:
        return False
    return p[0] == g[0] and p[1] == g[1]

def calculate_metrics(gold_data, pred_data, match_function):
    if not pred_data:
        precision = 0.0
    else:
        correct_pred = 0
        for p in pred_data:
            for g in gold_data:
                if match_function(p, g):
                    correct_pred += 1
                    break
        precision = correct_pred / len(pred_data)
    
    if not gold_data:
        recall = 0.0
    else:
        correct_recall = 0
        for g in gold_data:
            for p in pred_data:
                if match_function(p, g):
                    correct_recall += 1
                    break
        recall = correct_recall / len(gold_data)
    
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    
    return {
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4)
    }

def evaluate_matskraft(gold_csv_path, pred_csv_path):
    # Load data
    gold_df = pd.read_csv(gold_csv_path)
    pred_df = pd.read_csv(pred_csv_path)
    
    # Prepare data
    gold_df = prepare_data(gold_df)
    pred_df = prepare_data(pred_df)
    
    # Create and deduplicate composition data
    gold_comp = deduplicate_data(create_comp_list(gold_df), "composition")
    pred_comp = deduplicate_data(create_comp_list(pred_df), "composition")
    
    # Create property data
    gold_prop = create_prop_list(gold_df)
    pred_prop = create_prop_list(pred_df)
    
    # Create and deduplicate composition+property data
    gold_comp_prop = deduplicate_data(create_comp_and_prop_list(gold_df), "composition_property")
    pred_comp_prop = deduplicate_data(create_comp_and_prop_list(pred_df), "composition_property")
    
    # Calculate metrics
    comp_metrics = calculate_metrics(gold_comp, pred_comp, match_tuple_comp)
    prop_metrics = calculate_metrics(gold_prop, pred_prop, match_tuple_prop)
    comp_prop_metrics = calculate_metrics(gold_comp_prop, pred_comp_prop, match_tuple_comp_and_prop)
    
    # Print results
    print(f"COMPOSITION: P={comp_metrics['precision']:.4f}, R={comp_metrics['recall']:.4f}, F1={comp_metrics['f1']:.4f}")
    print(f"PROPERTY: P={prop_metrics['precision']:.4f}, R={prop_metrics['recall']:.4f}, F1={prop_metrics['f1']:.4f}")
    print(f"COMPOSITION_PROPERTY: P={comp_prop_metrics['precision']:.4f}, R={comp_prop_metrics['recall']:.4f}, F1={comp_prop_metrics['f1']:.4f}")
    
    return {
        'composition': comp_metrics,
        'property': prop_metrics,
        'composition_property': comp_prop_metrics
    }

In [3]:
results = evaluate_matskraft('gold_database.csv', 'pred_database.csv')

COMPOSITION: P=0.8147, R=0.6648, F1=0.7322
PROPERTY: P=0.9122, R=0.8737, F1=0.8926
COMPOSITION_PROPERTY: P=0.7808, R=0.6126, F1=0.6866
